# Mr.X R-GNN - Stage X1 BC logger for Kaggle

Notebook per Kaggle Notebook, pronto per **Save & Commit**.

Obiettivo: generare un dataset di behavioural cloning per una futura GNN di Mr.X. Il teacher e' `MrxEngine` MCTS; l'opponente e' la GNN detective PPO finale congelata.

Output principali in `/kaggle/working/mrx_bc_logs/...`, quindi inclusi negli output del commit Kaggle:
- `samples_part_*.parquet`: metadati leggibili, azione scelta, diagnostica MCTS, reward e return-to-go.
- `tensors_part_*.npz`: `node_dyn`, `glob`, `legal_action_mask`, `target_action`.
- `static_graph.npz`: adiacenza relazionale e feature statiche nodo.
- `manifest.json`: configurazione e statistiche della run.

Questo e' un logger, non un trainer. Questa versione e' configurata per un logging serio da 5000 game.

## 1. Setup Kaggle

In [ ]:
%%bash
set -e
cd /kaggle/working
python -m pip install -q networkx numpy pandas tqdm scipy pyarrow matplotlib || true

REPO_DIR=/kaggle/working/scotland_yard
if [ ! -d "$REPO_DIR" ]; then
  CANDIDATE=$(find /kaggle/input -maxdepth 4 -type d -name scotland_yard | head -n 1 || true)
  if [ -n "$CANDIDATE" ]; then
    echo "Copying repo from Kaggle input: $CANDIDATE"
    cp -R "$CANDIDATE" "$REPO_DIR"
  else
    echo "Cloning repo from GitHub"
    git clone --depth 1 https://github.com/Jacopo888/scotland_yard.git "$REPO_DIR"
  fi
fi
ls "$REPO_DIR"


In [ ]:
import glob
import os

KAGGLE_INPUT = '/kaggle/input'
KAGGLE_WORKING = '/kaggle/working'
MRX_LOG_ROOT = os.path.join(KAGGLE_WORKING, 'mrx_bc_logs')
os.makedirs(MRX_LOG_ROOT, exist_ok=True)

checkpoint_candidates = []
checkpoint_candidates += sorted(glob.glob(os.path.join(KAGGLE_INPUT, '**', 'rgnn_ppo_best_*.pt'), recursive=True))
checkpoint_candidates += sorted(glob.glob('/kaggle/working/scotland_yard/Notebook/rgnn_ppo_best_*.pt'))
checkpoint_candidates += sorted(glob.glob('/kaggle/working/scotland_yard/rgnn_ppo_best_*.pt'))

assert checkpoint_candidates, (
    'Nessun checkpoint detective PPO trovato. Aggiungi a Kaggle un input dataset/model '
    'che contenga rgnn_ppo_best_*.pt, oppure includilo nel repo sotto Notebook/.'
)

checkpoint_candidates = sorted(set(checkpoint_candidates), key=lambda p: os.path.basename(p))
DETECTIVE_CKPT_PATH = checkpoint_candidates[-1]
print('Detective checkpoint:', DETECTIVE_CKPT_PATH)
print('Mr.X log root:', MRX_LOG_ROOT)


## 2. Imports

In [ ]:
import json
import math
import os
import random
import sys
import time
from collections import Counter, defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch

REPO_DIR = '/kaggle/working/scotland_yard'
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from detective_engine import DetectiveEngine
from game import ADJ, SHORTEST_PATH_TENSOR, Game
from gnn_detective_engine import GNNDetectiveEngine
import mrx_engine
from mrx_engine import MrxEngine
from utility import _min_detective_distance

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))


## 3. Config + static graph

In [ ]:
N_NODES = 199
RELATIONS = ('taxi', 'bus', 'underground', 'water')
REL2IDX = {r: i for i, r in enumerate(RELATIONS)}
IDX2REL = {i: r for r, i in REL2IDX.items()}
TICKET_VOCAB = {'': 0, 'taxi': 1, 'bus': 2, 'underground': 3, 'water': 4, 'blocked': 0}
MAX_TURN = 22
REVEAL_TURNS = (3, 8, 13, 18)

# Serious logging run for Kaggle Save & Commit.
N_GAMES = 5000
SEED_BASE = 132159
GAMES_PER_CHUNK = 250

# Sample a teacher budget per Mr.X decision.
MCTS_BUDGETS = [(8, 8), (15, 25), (30, 50), (50, 80)]
MCTS_WEIGHTS = np.array([0.25, 0.45, 0.25, 0.05], dtype=np.float64)
MCTS_WEIGHTS /= MCTS_WEIGHTS.sum()

# Mr.X-centric reward for value targets. Policy BC uses target_action directly.
GAMMA = 0.95
R_TIMEOUT = +10.0
R_CAPTURE = -10.0
R_STEP = +0.05
R_DIST_COEF = +0.10

RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
OUT_DIR = os.path.join(MRX_LOG_ROOT, f'mrx_bc_{RUN_TAG}')
os.makedirs(OUT_DIR, exist_ok=True)
print('Output dir:', OUT_DIR)


def is_reveal_turn(turn):
    return int(turn) in REVEAL_TURNS


def turns_to_next_reveal(turn):
    for reveal_turn in REVEAL_TURNS:
        if reveal_turn > turn:
            return reveal_turn - turn
    return -1


def build_dense_adj():
    adj = np.zeros((len(RELATIONS), N_NODES, N_NODES), dtype=np.float32)
    for v_str, neigh in ADJ.items():
        v = int(v_str) - 1
        for u_str, types in neigh.items():
            u = int(u_str) - 1
            for vehicle in types:
                if vehicle in REL2IDX:
                    adj[REL2IDX[vehicle], v, u] = 1.0
    deg = adj.sum(axis=2, keepdims=True)
    deg[deg == 0] = 1.0
    return adj / deg


def build_node_static_features():
    deg = np.zeros((N_NODES, len(RELATIONS)), dtype=np.float32)
    for v_str, neigh in ADJ.items():
        v = int(v_str) - 1
        for _, types in neigh.items():
            for vehicle in types:
                if vehicle in REL2IDX:
                    deg[v, REL2IDX[vehicle]] += 1.0
    has_underground = (deg[:, REL2IDX['underground']] > 0).astype(np.float32)
    return np.concatenate([deg, has_underground[:, None]], axis=1).astype(np.float32)


DENSE_ADJ = build_dense_adj()
NODE_STATIC = build_node_static_features()
MRX_NODE_DYN_DIM = 14
MRX_GLOBAL_DIM = 28

np.savez_compressed(
    os.path.join(OUT_DIR, 'static_graph.npz'),
    dense_adj=DENSE_ADJ.astype(np.float32),
    node_static=NODE_STATIC.astype(np.float32),
    relations=np.array(RELATIONS),
    node_dyn_dim=np.array([MRX_NODE_DYN_DIM], dtype=np.int64),
    global_dim=np.array([MRX_GLOBAL_DIM], dtype=np.int64),
)
print('Static graph saved.')


## 4. Feature, action, and MCTS helpers

In [ ]:
_SP_ANY_NP = SHORTEST_PATH_TENSOR[7, :N_NODES, :N_NODES].astype(np.float32)


def json_default(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    raise TypeError(f'Object of type {type(obj)} is not JSON serializable')


def dumps(obj):
    return json.dumps(obj, default=json_default, separators=(',', ':'))


def flat_action(destination, ticket):
    if ticket not in REL2IDX:
        return -1
    return (int(destination) - 1) * len(RELATIONS) + REL2IDX[ticket]


def build_mrx_legal_actions(game):
    mask = np.zeros((N_NODES, len(RELATIONS)), dtype=np.bool_)
    actions = []
    detective_set = set(game.detectives_pos)
    for ticket, nodes in game.find_legal_moves_x().items():
        if ticket not in REL2IDX:
            continue
        rel_idx = REL2IDX[ticket]
        for destination in nodes:
            dst_idx = int(destination) - 1
            mask[dst_idx, rel_idx] = True
            actions.append({
                'destination': str(destination),
                'ticket': ticket,
                'flat': int(dst_idx * len(RELATIONS) + rel_idx),
                'captures_detective': bool(destination in detective_set),
            })
    return mask, actions


def build_mrx_input(game, engine, last_mrx_ticket, revealed_positions):
    belief = engine.belief_state.astype(np.float32)
    mrx_idx = int(game.mrx_pos) - 1
    det_pos_arr = np.array([int(p) - 1 for p in game.detectives_pos], dtype=np.int64)

    is_mrx = np.zeros((N_NODES, 1), dtype=np.float32)
    is_mrx[mrx_idx, 0] = 1.0

    is_det = np.zeros((5, N_NODES), dtype=np.float32)
    for k, p in enumerate(det_pos_arr):
        is_det[k, p] = 1.0

    revealed = np.zeros((N_NODES, 1), dtype=np.float32)
    for p in revealed_positions:
        revealed[int(p) - 1, 0] = 1.0

    dist_per_det = np.stack([_SP_ANY_NP[p] for p in det_pos_arr], axis=1).astype(np.float32)
    min_dist = dist_per_det.min(axis=1, keepdims=True).astype(np.float32)

    node_dyn = np.concatenate(
        [
            is_mrx,
            is_det.T,
            belief[:, None],
            revealed,
            min_dist,
            dist_per_det,
        ],
        axis=1,
    ).astype(np.float32)
    assert node_dyn.shape == (N_NODES, MRX_NODE_DYN_DIM), node_dyn.shape

    last_ticket_oh = np.zeros(5, dtype=np.float32)
    last_ticket_oh[TICKET_VOCAB.get(last_mrx_ticket, 0)] = 1.0
    all_det_tickets = np.array(
        [[t['taxi'], t['bus'], t['underground']] for t in game.detective_tickets],
        dtype=np.float32,
    ) / 10.0
    glob = np.concatenate(
        [
            np.array(
                [
                    game.turn / MAX_TURN,
                    turns_to_next_reveal(game.turn) / 5.0,
                    float(is_reveal_turn(game.turn)),
                    float(is_reveal_turn(game.turn + 1)),
                ],
                dtype=np.float32,
            ),
            np.array([game.mrx_tickets[v] for v in RELATIONS], dtype=np.float32) / 10.0,
            all_det_tickets.reshape(-1),
            last_ticket_oh,
        ]
    ).astype(np.float32)
    assert glob.shape == (MRX_GLOBAL_DIM,), glob.shape
    return node_dyn, glob


def extract_mcts_diagnostics(mrx_policy, selected_destination, selected_ticket):
    children = list(mrx_policy.root.children)
    child_rows = []
    visits_by_flat = defaultdict(int)
    selected = None
    for child in children:
        dst = str(child.game_status.mrx_pos)
        ticket = child.ticket
        action_flat = flat_action(dst, ticket)
        visits = int(child.visits)
        score = float(child.score)
        mean_score = score / visits if visits else None
        row = {
            'destination': dst,
            'ticket': ticket,
            'flat': int(action_flat),
            'visits': visits,
            'score': score,
            'mean_score': mean_score,
            'is_terminal': bool(child.is_terminal),
        }
        child_rows.append(row)
        if action_flat >= 0:
            visits_by_flat[int(action_flat)] += visits
        if dst == str(selected_destination) and ticket == selected_ticket:
            selected = row

    total_visits = sum(visits_by_flat.values())
    selected_flat = flat_action(selected_destination, selected_ticket)
    if total_visits > 0:
        sparse_policy = [
            {'flat': int(k), 'prob': float(v / total_visits)}
            for k, v in sorted(visits_by_flat.items())
            if v > 0
        ]
    elif selected_flat >= 0:
        sparse_policy = [{'flat': int(selected_flat), 'prob': 1.0}]
    else:
        sparse_policy = []

    entropy = 0.0
    for item in sparse_policy:
        p = item['prob']
        if p > 0:
            entropy -= p * math.log(p)
    return child_rows, sparse_policy, entropy, selected


def discounted_returns(rewards, gamma=GAMMA):
    out = [0.0] * len(rewards)
    running = 0.0
    for i in range(len(rewards) - 1, -1, -1):
        running = float(rewards[i]) + gamma * running
        out[i] = running
    return out


## 5. Logger loop

In [ ]:
detective_policy = GNNDetectiveEngine(checkpoint_path=DETECTIVE_CKPT_PATH, device=DEVICE)
print('Loaded detective checkpoint file:', detective_policy.checkpoint_path)
print('Loaded detective GNN metadata:')
print('  note: source_checkpoint below is checkpoint provenance, not the file loaded by this logger')
for k, v in detective_policy.metadata.items():
    print(f'  {k}: {v}')

part_id = 0
next_sample_id = 0
records = []
node_dyn_buf = []
glob_buf = []
legal_mask_buf = []
target_buf = []
part_paths = []
game_stats = []


def set_mrx_strength(explorations, simulations):
    mrx_engine.NUM_EXPLORATIONS = int(explorations)
    mrx_engine.NUM_SIMULATIONS = int(simulations)


def sample_mcts_budget():
    idx = int(np.random.choice(len(MCTS_BUDGETS), p=MCTS_WEIGHTS))
    return MCTS_BUDGETS[idx]


def play_detective_phase(game, engine, policy):
    for detective_id in range(game.num_detectives):
        policy.play_detective_turn(game, engine.belief_state, detective_id)
        if game.check_victory(silent=True):
            return True
        engine.kalman_filter()
    game.detectives_moves.append(game.detectives_pos[:])
    return False


def flush_part():
    global part_id, records, node_dyn_buf, glob_buf, legal_mask_buf, target_buf
    if not records:
        return None
    base = f'part_{part_id:03d}'
    parquet_path = os.path.join(OUT_DIR, f'samples_{base}.parquet')
    npz_path = os.path.join(OUT_DIR, f'tensors_{base}.npz')

    df = pd.DataFrame(records)
    df.to_parquet(parquet_path, index=False)
    np.savez_compressed(
        npz_path,
        node_dyn=np.asarray(node_dyn_buf, dtype=np.float16),
        glob=np.asarray(glob_buf, dtype=np.float32),
        legal_action_mask=np.asarray(legal_mask_buf, dtype=np.bool_),
        target_action=np.asarray(target_buf, dtype=np.int64),
    )
    part_paths.append({'samples': parquet_path, 'tensors': npz_path, 'rows': int(len(df))})
    print(f'Saved {base}: {len(df)} rows')

    part_id += 1
    records = []
    node_dyn_buf = []
    glob_buf = []
    legal_mask_buf = []
    target_buf = []
    return parquet_path, npz_path


def run_logged_game(game_id, seed):
    global next_sample_id
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    detective_policy.reset()
    game = Game()
    engine = DetectiveEngine(game.detectives_pos)
    mrx_policy = MrxEngine(game, engine)

    last_mrx_ticket = ''
    revealed_positions = []
    local_record_indices = []
    blocked_decisions = 0
    mrx_decision_id = 0
    budget_counter = Counter()

    while True:
        if game.check_victory(silent=True):
            break

        if play_detective_phase(game, engine, detective_policy):
            break

        legal_mask, legal_actions = build_mrx_legal_actions(game)
        if not legal_actions:
            blocked_decisions += 1
            game.x_automated_turn(game.mrx_pos, None)
            if game.check_victory(silent=True):
                break
            engine.update_belief_after_mrx_move(None)
            if is_reveal_turn(game.turn):
                engine.mrx_is_spotted(game.mrx_pos)
                revealed_positions.append(int(game.mrx_pos))
            detective_policy.observe_mrx_move(game, None)
            mrx_decision_id += 1
            continue

        mrx_explo, mrx_sims = sample_mcts_budget()
        budget_counter[(mrx_explo, mrx_sims)] += 1
        set_mrx_strength(mrx_explo, mrx_sims)

        node_dyn, glob_features = build_mrx_input(
            game,
            engine,
            last_mrx_ticket=last_mrx_ticket,
            revealed_positions=tuple(revealed_positions),
        )

        best_pos, best_ticket = mrx_policy.search()

        if best_ticket is None:
            blocked_decisions += 1
            game.x_automated_turn(best_pos, best_ticket)
            if game.check_victory(silent=True):
                break
            engine.update_belief_after_mrx_move(best_ticket)
            if is_reveal_turn(game.turn):
                engine.mrx_is_spotted(game.mrx_pos)
                revealed_positions.append(int(game.mrx_pos))
            detective_policy.observe_mrx_move(game, best_ticket)
            mrx_decision_id += 1
            continue

        child_rows, sparse_policy, visit_entropy, selected_child = extract_mcts_diagnostics(
            mrx_policy, best_pos, best_ticket
        )
        target = flat_action(best_pos, best_ticket)
        target_ok = bool(legal_mask.reshape(-1)[target]) if target >= 0 else False
        if not target_ok:
            raise AssertionError(
                f'MCTS target not in legal mask: game={game_id}, turn={game.turn}, '
                f'target=({best_pos}, {best_ticket}), legal={legal_actions}'
            )

        belief_argmax = int(np.argmax(engine.belief_state)) + 1
        belief_max = float(np.max(engine.belief_state))
        sample_id = next_sample_id
        next_sample_id += 1

        record = {
            'sample_id': int(sample_id),
            'game_id': int(game_id),
            'decision_id': int(mrx_decision_id),
            'turn_before_mrx': int(game.turn),
            'mrx_pos_before': str(game.mrx_pos),
            'detectives_pos_json': dumps([str(p) for p in game.detectives_pos]),
            'mrx_tickets_json': dumps(game.mrx_tickets),
            'detective_tickets_json': dumps(game.detective_tickets),
            'last_mrx_ticket': str(last_mrx_ticket),
            'revealed_positions_json': dumps(revealed_positions),
            'belief_argmax': int(belief_argmax),
            'belief_max': float(belief_max),
            'n_legal_actions': int(len(legal_actions)),
            'legal_actions_json': dumps(legal_actions),
            'mcts_explorations': int(mrx_explo),
            'mcts_simulations': int(mrx_sims),
            'target_destination': str(best_pos),
            'target_ticket': str(best_ticket),
            'target_action': int(target),
            'target_in_legal_mask': bool(target_ok),
            'mcts_children_json': dumps(child_rows),
            'mcts_visit_policy_json': dumps(sparse_policy),
            'mcts_visit_entropy': float(visit_entropy),
            'selected_child_visits': int(selected_child['visits']) if selected_child else 0,
            'selected_child_mean_score': float(selected_child['mean_score']) if selected_child and selected_child['mean_score'] is not None else np.nan,
            'reward': 0.0,
            'return_to_go': np.nan,
            'winner': -1,
            'final_turn': -1,
            'blocked_decisions_in_game': 0,
        }

        records.append(record)
        local_record_indices.append(len(records) - 1)
        node_dyn_buf.append(node_dyn)
        glob_buf.append(glob_features)
        legal_mask_buf.append(legal_mask)
        target_buf.append(target)

        game.x_automated_turn(best_pos, best_ticket)
        min_dist_after = _min_detective_distance(game.detectives_pos, game.mrx_pos)
        records[local_record_indices[-1]]['reward'] = float(R_STEP + R_DIST_COEF * min_dist_after)
        records[local_record_indices[-1]]['mrx_pos_after'] = str(game.mrx_pos)
        records[local_record_indices[-1]]['min_dist_after_mrx'] = int(min_dist_after)

        if game.check_victory(silent=True):
            break

        engine.update_belief_after_mrx_move(best_ticket)
        if is_reveal_turn(game.turn):
            engine.mrx_is_spotted(game.mrx_pos)
            revealed_positions.append(int(game.mrx_pos))
        detective_policy.observe_mrx_move(game, best_ticket)
        last_mrx_ticket = best_ticket
        mrx_decision_id += 1

    game.check_victory(silent=True)
    winner = int(game.winner)
    if local_record_indices:
        terminal_bonus = R_TIMEOUT if winner == 1 else R_CAPTURE
        records[local_record_indices[-1]]['reward'] = float(records[local_record_indices[-1]]['reward'] + terminal_bonus)
        rewards = [records[idx]['reward'] for idx in local_record_indices]
        returns = discounted_returns(rewards)
        for idx, rtg in zip(local_record_indices, returns):
            records[idx]['return_to_go'] = float(rtg)
            records[idx]['winner'] = winner
            records[idx]['final_turn'] = int(game.turn)
            records[idx]['blocked_decisions_in_game'] = int(blocked_decisions)

    return {
        'game_id': int(game_id),
        'seed': int(seed),
        'winner': winner,
        'mrx_win': int(winner == 1),
        'final_turn': int(game.turn),
        'logged_decisions': int(len(local_record_indices)),
        'blocked_decisions': int(blocked_decisions),
        'budget_counts': {f'{k[0]}/{k[1]}': int(v) for k, v in budget_counter.items()},
    }


started = time.time()
for game_id in tqdm(range(N_GAMES), desc='Logging games'):
    stats = run_logged_game(game_id=game_id, seed=SEED_BASE + game_id)
    game_stats.append(stats)
    if (game_id + 1) % GAMES_PER_CHUNK == 0:
        flush_part()

flush_part()
elapsed = time.time() - started
print(f'Logged {next_sample_id} Mr.X decisions from {N_GAMES} games in {elapsed/60:.1f} min')


## 6. Manifest + sanity checks

In [ ]:
stats_df = pd.DataFrame(game_stats)
sample_parts = [pd.read_parquet(p['samples']) for p in part_paths]
samples_df = pd.concat(sample_parts, ignore_index=True) if sample_parts else pd.DataFrame()

ticket_counts = samples_df['target_ticket'].value_counts().to_dict() if not samples_df.empty else {}
budget_counts = Counter()
for row in game_stats:
    budget_counts.update(row['budget_counts'])

sanity = {
    'n_games': int(N_GAMES),
    'n_samples': int(len(samples_df)),
    'mrx_winrate': float(stats_df['mrx_win'].mean()) if len(stats_df) else None,
    'avg_final_turn': float(stats_df['final_turn'].mean()) if len(stats_df) else None,
    'avg_logged_decisions': float(stats_df['logged_decisions'].mean()) if len(stats_df) else None,
    'blocked_decisions_total': int(stats_df['blocked_decisions'].sum()) if len(stats_df) else 0,
    'target_in_legal_mask_rate': float(samples_df['target_in_legal_mask'].mean()) if len(samples_df) else None,
    'ticket_counts': {str(k): int(v) for k, v in ticket_counts.items()},
    'budget_counts': {str(k): int(v) for k, v in budget_counts.items()},
}

manifest = {
    'run_tag': RUN_TAG,
    'generated_at': datetime.now().isoformat(),
    'repo_dir': REPO_DIR,
    'detective_checkpoint': DETECTIVE_CKPT_PATH,
    'detective_checkpoint_metadata': detective_policy.metadata,
    'out_dir': OUT_DIR,
    'parts': part_paths,
    'config': {
        'n_games': N_GAMES,
        'seed_base': SEED_BASE,
        'games_per_chunk': GAMES_PER_CHUNK,
        'mcts_budgets': MCTS_BUDGETS,
        'mcts_weights': MCTS_WEIGHTS.tolist(),
        'gamma': GAMMA,
        'reward': {
            'timeout': R_TIMEOUT,
            'capture': R_CAPTURE,
            'step': R_STEP,
            'distance_coef': R_DIST_COEF,
        },
        'node_dyn_dim': MRX_NODE_DYN_DIM,
        'global_dim': MRX_GLOBAL_DIM,
        'relations': RELATIONS,
    },
    'sanity': sanity,
    'elapsed_seconds': elapsed,
}

manifest_path = os.path.join(OUT_DIR, 'manifest.json')
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, default=json_default)

stats_path = os.path.join(OUT_DIR, 'game_stats.parquet')
stats_df.to_parquet(stats_path, index=False)

print('Manifest:', manifest_path)
print('Game stats:', stats_path)
print(json.dumps(sanity, indent=2))

assert sanity['target_in_legal_mask_rate'] == 1.0, 'Some MCTS targets are outside the legal mask.'
assert sanity['n_samples'] > 0, 'No Mr.X decisions were logged.'
display(samples_df.head(10))


## 7. Quick inspection

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
stats_df['final_turn'].hist(ax=axes[0], bins=range(0, MAX_TURN + 2))
axes[0].set_title('Final turn')
axes[0].set_xlabel('turn')

samples_df['target_ticket'].value_counts().reindex(RELATIONS, fill_value=0).plot(kind='bar', ax=axes[1])
axes[1].set_title('MCTS target ticket')

samples_df['mcts_visit_entropy'].hist(ax=axes[2], bins=30)
axes[2].set_title('MCTS visit entropy')
axes[2].set_xlabel('entropy')

plt.tight_layout()
plt.show()

print('Rows per part:')
for p in part_paths:
    print(p)

print('\nDataset ready in:', OUT_DIR)
